# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# The file is not UTF-8 encoded, so we need encoding="latin-1"
df = pd.read_csv("data/AviationData.csv", encoding="latin-1", low_memory=False)
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head()

Rows: 88,889   Columns: 31


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


**Datatypes and non-null counts**

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

**Missing values (NaNs) per column**

In [4]:
nan_summary = pd.DataFrame({"n_missing": df.isna().sum(),
                            "pct_missing": (df.isna().mean() * 100).round(1)})
nan_summary.sort_values("n_missing", ascending=False)

,n_missing,pct_missing
Schedule,76307,85.8
Air.carrier,72241,81.3
FAR.Description,56866,64.0
Aircraft.Category,56602,63.7
Longitude,54516,61.3
Latitude,54507,61.3
Airport.Code,38757,43.6
Airport.Name,36185,40.7
Broad.phase.of.flight,27165,30.6
Publication.Date,13771,15.5


**Summary statistics**

In [5]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [6]:
df.describe(exclude="number").T

,count,unique,top,freq
Event.Id,88889,87951,20001214X45071,3
Investigation.Type,88889,2,Accident,85015
Accident.Number,88889,88863,ERA22LA103,2
Event.Date,88889,14782,1982-05-16,25
Location,88837,27758,"ANCHORAGE, AK",434
Country,88663,219,United States,82248
Latitude,34382,25589,332739N,19
Longitude,34373,27154,0112457W,24
Airport.Code,50132,10374,NONE,1488
Airport.Name,52704,24870,Private,240


**Observations from the inspection**

- The raw file has one row **per aircraft involved** in an event. `Event.Id` repeats for some rows - a quick look at these rows shows they are *different aircraft in the same event* (e.g. mid-air collisions: accident numbers end in `A`/`B`, registrations differ). They are **not duplicates**, so we do **not** drop on `Event.Id`. We verify below that there are no fully duplicated rows.
- The aircraft damage column is spelled `Aircraft.damage` (lower-case *d*) in the file; we will rename it to `Aircraft.Damage`.
- `Aircraft.Category` is missing for ~64% of rows, so filtering it naively would throw away most of the data. We will impute it (see below).
- `Make` has inconsistent capitalisation and naming (`Cessna` vs `CESSNA`, `Robinson` vs `Robinson Helicopter Co.`).
- Injury count columns are NaN in ~13% of rows. NaN is used where the count is zero (checked below).

In [7]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Rows sharing an Event.Id with another row:", df["Event.Id"].duplicated(keep=False).sum())
df[df["Event.Id"].duplicated(keep=False)].sort_values("Event.Id")[
    ["Event.Id", "Accident.Number", "Registration.Number", "Make", "Model"]].head(6)

Fully duplicated rows: 0
Rows sharing an Event.Id with another row: 1874


,Event.Id,Accident.Number,Registration.Number,Make,Model
45562,20001204X00086,LAX99LA086B,N770RF,Coelho,RV4
45564,20001204X00086,LAX99LA086A,N6979G,Cessna,150L
45703,20001205X00276,CHI99IA100B,N106CK,Lockheed,L-1O11-385-1-15
45704,20001205X00276,CHI99IA100A,N40061,Mcdonnell Douglas,DC-10
45716,20001205X00305,DEN99LA047B,N328BA,Swearingen,SA226TC
45719,20001205X00305,DEN99LA047A,N151SA,Swearingen,SA226TC


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Step 1 - dates.** Convert `Event.Date` to a datetime.

In [8]:
df["Event.Date"] = pd.to_datetime(df["Event.Date"])
print(df["Event.Date"].min().date(), "->", df["Event.Date"].max().date())

1948-10-24 -> 2022-12-29


**Step 2 - `Aircraft.Category`: inspect and impute.**

Only ~36% of rows have a category. Missing categories are mostly *old* records, so dropping them would remove most of the data. 
The same make + model always has the same category (a Cessna 172 is always an airplane), so we can **fill missing categories from the most common category of rows with the same Make + Model**. Rows whose make/model never appears with a known category stay missing and will be dropped by the filter.

In [9]:
print(df["Aircraft.Category"].value_counts(dropna=False))

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


In [10]:
# Treat "Unknown"/"UNK" as missing
df["Aircraft.Category"] = df["Aircraft.Category"].where(~df["Aircraft.Category"].isin(["Unknown", "UNK"]))

# Temporary lookup key: upper-cased Make|Model (only defined when both exist)
df["_key"] = (df["Make"].str.upper().str.strip() + "|" + df["Model"].str.upper().str.strip())

# Most common known category for every Make|Model
lookup = (df.dropna(subset=["Aircraft.Category", "_key"])
            .groupby("_key")["Aircraft.Category"]
            .agg(lambda s: s.value_counts().idxmax()))

missing_before = df["Aircraft.Category"].isna().sum()
df["Aircraft.Category"] = df["Aircraft.Category"].fillna(df["_key"].map(lookup))
print(f"Missing categories: {missing_before:,} -> {df['Aircraft.Category'].isna().sum():,}")
print(df["Aircraft.Category"].value_counts(dropna=False).head(6))

Missing categories: 56,618 -> 11,918
Aircraft.Category
Airplane      67957
NaN           11918
Helicopter     7246
Glider          881
Balloon         376
Gyrocraft       200
Name: count, dtype: int64


**Step 3 - `Amateur.Built`.** The client only wants professional builds (`No`). The 102 NaNs cannot be verified as professional builds, so they are dropped by the filter.

**Step 4 - `Event.Date`.** Keep events from 1983-01-01 onward (max 40-year make/model lifetime).

We apply all three filters and print how many rows remain after each step.

In [11]:
print("Amateur.Built values:", df["Amateur.Built"].value_counts(dropna=False).to_dict())

n0 = len(df)
df = df[df["Aircraft.Category"] == "Airplane"]
print(f"Airplanes only:            {len(df):>7,}  (removed {n0-len(df):,})"); n0 = len(df)

df = df[df["Amateur.Built"] == "No"]
print(f"Professional builds only:  {len(df):>7,}  (removed {n0-len(df):,})"); n0 = len(df)

df = df[df["Event.Date"] >= "1983-01-01"]
print(f"Events from 1983 onward:   {len(df):>7,}  (removed {n0-len(df):,})")

df = df.copy()   # avoid SettingWithCopy issues later

Amateur.Built values: {'No': 80312, 'Yes': 8475, nan: 102}
Airplanes only:             67,957  (removed 20,932)
Professional builds only:   64,380  (removed 3,577)
Events from 1983 onward:    61,385  (removed 2,995)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Cleaning assumptions - injury columns**

- The dataset has four occupant counts: `Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`. 
- We check that a NaN really means *zero* before filling it: rows with `Injury.Severity == "Non-Fatal"` almost always have NaN fatal counts.

In [12]:
ic = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]

# Sanity check: when is Total.Fatal.Injuries missing?
print(df.loc[df["Total.Fatal.Injuries"].isna(), "Injury.Severity"].value_counts(dropna=False).head(5))
print("\nRows with severity 'Fatal(n)' but NaN fatal count:",
      (df["Injury.Severity"].str.startswith("Fatal", na=False) & df["Total.Fatal.Injuries"].isna()).sum())

Injury.Severity
Non-Fatal      7439
Incident        397
Unavailable      39
Name: count, dtype: int64

Rows with severity 'Fatal(n)' but NaN fatal count: 0


NaN fatal counts occur for non-fatal events, and never for a fatal event, so **NaN = 0** is a sound assumption for an individual injury column.

**Derived columns**
- `Total.Passengers` = fatal + serious + minor + uninjured. The data does not separate crew from passengers, so this is the *total number of occupants* on board - our estimate of passengers.
- `Total.Fatal.Serious` = fatal + serious injuries.
- `Fatal.Serious.Fraction` = `Total.Fatal.Serious / Total.Passengers`, the estimated likelihood that an occupant was fatally/seriously injured in the event (between 0 and 1).

Rows where **all four** counts are NaN carry no occupant information, and rows with 0 occupants have no defined fraction, so both are dropped.

In [13]:
n0 = len(df)
df = df.dropna(subset=ic, how="all")                 # no occupant information at all
print(f"Dropped {n0-len(df):,} rows with all injury counts missing")

df[ic] = df[ic].fillna(0)                             # NaN means 0 (see check above)

df["Total.Passengers"] = df[ic].sum(axis=1)
n0 = len(df)
df = df[df["Total.Passengers"] > 0].copy()            # fraction undefined with 0 occupants
print(f"Dropped {n0-len(df):,} rows with 0 occupants")

df["Total.Fatal.Serious"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["Fatal.Serious.Fraction"] = df["Total.Fatal.Serious"] / df["Total.Passengers"]

print(f"\nRows remaining: {len(df):,}")
df[["Total.Passengers", "Total.Fatal.Serious", "Fatal.Serious.Fraction"]].describe()

Dropped 114 rows with all injury counts missing
Dropped 958 rows with 0 occupants

Rows remaining: 60,313


,Total.Passengers,Total.Fatal.Serious,Fatal.Serious.Fraction
count,60313.000000,60313.000000,60313.000000
mean,6.408983,0.789349,0.269156
std,28.534794,5.194705,0.428019
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,0.666667
max,588.000000,295.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [14]:
# The raw column is spelled "Aircraft.damage" - rename to match the instructions
df = df.rename(columns={"Aircraft.damage": "Aircraft.Damage"})
df["Aircraft.Damage"].value_counts(dropna=False)

Aircraft.Damage
Substantial    45142
Destroyed      11954
NaN             1587
Minor           1556
Unknown           74
Name: count, dtype: int64

**Cleaning tasks for `Aircraft.Damage`**
1. Strip whitespace / standardise capitalisation.
2. `"Unknown"` is not information about damage, so it is converted to NaN. 
3. The remaining NaNs (~3-4%) are **left as NaN** rather than imputed: we cannot know whether the aircraft was destroyed. Because the derived flag will also be NaN for these rows, they are automatically excluded from every *destroyed-rate* calculation, but still contribute to the injury analysis.

**Derived column:** `Destroyed` = 1 if damage is `"Destroyed"`, 0 if `"Substantial"` or `"Minor"`, NaN if unknown.

In [15]:
df["Aircraft.Damage"] = df["Aircraft.Damage"].str.strip().str.title()
df["Aircraft.Damage"] = df["Aircraft.Damage"].where(df["Aircraft.Damage"] != "Unknown")

df["Destroyed"] = (df["Aircraft.Damage"] == "Destroyed").astype(float).where(df["Aircraft.Damage"].notna())

print(df["Aircraft.Damage"].value_counts(dropna=False))
print(f"\nOverall destroyed rate: {df['Destroyed'].mean():.1%}")

Aircraft.Damage
Substantial    45142
Destroyed      11954
NaN             1661
Minor           1556
Name: count, dtype: int64

Overall destroyed rate: 20.4%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks for `Make`**
1. Drop rows where `Make` is missing.
2. Upper-case, strip whitespace, remove `.` and `,`, collapse repeated spaces (`Cessna` / `CESSNA`).
3. Merge spelling / corporate-name variants of the same manufacturer, e.g. `CESSNA AIRCRAFT CO` -> `CESSNA`, `ROBINSON HELICOPTER COMPANY` -> `ROBINSON`, `DEHAVILLAND` -> `DE HAVILLAND`, `HAWKER BEECHCRAFT` -> `BEECH`, `AIRBUS INDUSTRIE` -> `AIRBUS`, `GATES LEARJET` -> `LEARJET`. This is done with regular-expression rules that match on the *start* of the name.
4. Strip trailing corporate suffixes (`INC`, `CORP`, `CORPORATION`, `CO`, `COMPANY`, `LTD`, `LLC`).
5. Keep only makes with **at least 50 events** so that each make has a reasonable number of samples.

In [16]:
n_before = df["Make"].nunique()
df = df.dropna(subset=["Make"]).copy()

make = (df["Make"].str.upper()
          .str.replace(r"[.,]", "", regex=True)
          .str.replace(r"\s+", " ", regex=True)
          .str.strip())

# (regex anchored at start of name, standard name)
make_rules = [
    (r"(?:NEW )?PIPER\b", "PIPER"),
    (r"CESSNA\b", "CESSNA"),
    (r"(?:HAWKER[ -])?BEECH", "BEECH"),
    (r"(?:THE )?BOEING(?: COMPANY)?$", "BOEING"),
    (r"MOONEY", "MOONEY"),
    (r"AIRBUS", "AIRBUS"),
    (r"EMBRAER", "EMBRAER"),
    (r"CIRRUS", "CIRRUS"),
    (r"DIAMOND AI", "DIAMOND"),
    (r"DE ?HAVILLAND", "DE HAVILLAND"),
    (r"AVIAT(?: AIRCRAFT)?(?: INC)?$", "AVIAT"),
    (r"ERCOUPE", "ERCOUPE"),
    (r"MCDONNELL[ -]DOUGLAS(?: AIRCRAFT)?(?: CO)?$", "MCDONNELL DOUGLAS"),
    (r"(?:GATES |BOMBARDIER )?LEARJET", "LEARJET"),
    (r"RAYTHEON", "RAYTHEON"),
    (r"(?:EADS )?SOCATA", "SOCATA"),
    (r"PILATUS", "PILATUS"),
    (r"SAAB", "SAAB"),
    (r"BOMBARDIER", "BOMBARDIER"),
    (r"SWEARINGEN", "SWEARINGEN"),
    (r"AIR TRACTOR", "AIR TRACTOR"),
    (r"ROBINSON HELICOPTER", "ROBINSON"),
    (r"GRUMMAN(?! AMERICAN)", "GRUMMAN"),        # Grumman, Grumman-Schweizer, Grumman Acft Eng ...
    (r"GRUMMAN AMERICAN", "GRUMMAN AMERICAN"),
    (r"ROCKWELL", "ROCKWELL"),
    (r"AMERICAN CHAMPION", "CHAMPION"),
]
for pattern, standard in make_rules:
    make = make.where(~make.str.match(pattern), standard)

make = make.str.replace(r"\s+(?:INC|CORP|CORPORATION|CO|COMPANY|LTD|LLC)$", "", regex=True)
df["Make"] = make

print(f"Unique makes: {n_before:,} (raw) -> {df['Make'].nunique():,} (cleaned)")
make_counts = df["Make"].value_counts()
make_counts.head(15)

Unique makes: 1,343 (raw) -> 890 (cleaned)


Make
CESSNA          25104
PIPER           13715
BEECH            4628
BOEING           1632
MOONEY           1245
GRUMMAN          1156
BELLANCA          919
AIR TRACTOR       854
CHAMPION          565
AERONCA           534
MAULE             495
CIRRUS            455
DE HAVILLAND      404
STINSON           392
LUSCOMBE          368
Name: count, dtype: int64

In [17]:
MIN_MAKE_EVENTS = 50
keep_makes = make_counts[make_counts >= MIN_MAKE_EVENTS].index

print(f"Makes with >= {MIN_MAKE_EVENTS} events: {len(keep_makes)} of {len(make_counts):,}")
print(f"Share of events retained:     {make_counts[keep_makes].sum() / len(df):.1%}")

df = df[df["Make"].isin(keep_makes)].copy()
print(f"Rows remaining: {len(df):,}")

Makes with >= 50 events: 49 of 890
Share of events retained:     95.3%
Rows remaining: 57,481


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [18]:
print("Missing Model:", df["Model"].isna().sum())
df = df.dropna(subset=["Model"]).copy()

# Standardise: upper-case, strip, collapse repeated whitespace
df["Model"] = df["Model"].str.upper().str.strip().str.replace(r"\s+", " ", regex=True)
print("Unique Model labels:", df["Model"].nunique())

Missing Model: 13
Unique Model labels: 2352


**Are model labels unique to each make?** Count how many different makes share each model label:

In [19]:
makes_per_model = df.groupby("Model")["Make"].nunique()
print(f"Model labels used by more than one make: {(makes_per_model > 1).sum()} of {len(makes_per_model)}")

# the most frequently reported ambiguous labels
shared = makes_per_model[makes_per_model > 1].index
ambiguous = (df[df["Model"].isin(shared)].groupby("Model")
               .agg(events=("Make", "size"), makes=("Make", lambda s: ", ".join(sorted(s.unique())[:4]))))
ambiguous.sort_values("events", ascending=False).head(8)

Model labels used by more than one make: 103 of 2352


,events,makes
Model,,
A36,438,"BEECH, RAYTHEON"
G-164A,424,"GRUMMAN, SCHWEIZER"
140,385,"CESSNA, EMBRAER"
G-164B,360,"GRUMMAN, SCHWEIZER"
7AC,274,"AERONCA, CHAMPION"
7GCBC,259,"BELLANCA, CHAMPION"
7ECA,248,"BELLANCA, CHAMPION"
58,240,"BEECH, RAYTHEON"


Model labels are **not unique to a make** (e.g. generic labels like `172` or `PA-28` can appear under more than one make), so grouping by `Model` alone would merge different aircraft. 

We therefore create the derived column **`Make.Model`** = `Make` + `" "` + `Model` as the unique identifier of a plane type.

> Note: variants such as `172M` and `172N` are kept as separate plane types - this keeps the identifier simple and faithful to the raw data (at the cost of splitting some large families into smaller samples; the analysis notebook handles this with a minimum-sample filter).

In [20]:
df["Make.Model"] = df["Make"] + " " + df["Model"]
print("Unique plane types (Make.Model):", df["Make.Model"].nunique())
df["Make.Model"].value_counts().head(8)

Unique plane types (Make.Model): 2464


Make.Model
CESSNA 152         2225
CESSNA 172         1637
CESSNA 172N        1091
PIPER PA-28-140     862
CESSNA 172M         758
CESSNA 150          748
CESSNA 172P         663
CESSNA 182          614
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [21]:
other_cols = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]
for c in other_cols:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).head(15).to_string(), "\n")

--- Engine.Type ---
Engine.Type
Reciprocating      49487
NaN                 3451
Turbo Prop          2311
Turbo Fan           1468
Unknown              423
Turbo Jet            313
Turbo Shaft           13
Geared Turbofan        1
UNK                    1 

--- Weather.Condition ---
Weather.Condition
VMC    50025
IMC     4496
NaN     2241
UNK      527
Unk      179 

--- Number.of.Engines ---
Number.of.Engines
1.0    46668
2.0     7709
NaN     2595
3.0      253
4.0      160
0.0       83 

--- Purpose.of.flight ---
Purpose.of.flight
Personal               33389
Instructional           7907
Unknown                 3465
NaN                     3429
Aerial Application      3154
Business                2710
Positioning              938
Other Work Use           502
Ferry                    488
Aerial Observation       370
Executive/corporate      294
Public Aircraft          259
Skydiving                172
Flight Test              121
Banner Tow                97 

--- Broad.phase.of.flight

**Cleaning tasks identified**

| Column | Problem | Action |
|---|---|---|
| `Engine.Type` | `Unknown`, `UNK`, `NONE`, `LR` (placeholders / meaningless); `Geared Turbofan` is a turbofan | Convert placeholders to NaN, merge `Geared Turbofan` -> `Turbo Fan`, fill NaN from the most common engine type of the same `Make.Model`, remaining -> `"Unknown"` |
| `Number.of.Engines` | `0` engines is impossible for an airplane (placeholder for NaN) | `0` -> NaN, fill from the most common value of the same `Make.Model`; remaining left as NaN |
| `Weather.Condition` | `UNK` / `Unk` / NaN all mean unknown | Standardise to `"Unknown"` (valid values: `VMC`, `IMC`) |
| `Purpose.of.flight` | Duplicated labels (`Air Race show` vs `Air Race/show`, `Public Aircraft - Federal/State/Local`, `Executive/corporate` vs `Business`) and many very rare categories | Merge duplicates, NaN -> `"Unknown"`, group categories with fewer than 50 events into `"Other"` |
| `Broad.phase.of.flight` | Many NaNs | NaN -> `"Unknown"` |

We do not drop any rows here.

In [22]:
def fill_with_group_mode(frame, col, key="Make.Model"):
    """Fill NaNs in `col` with the most common value of the same plane type."""
    mode = (frame.dropna(subset=[col]).groupby(key)[col]
                 .agg(lambda s: s.value_counts().idxmax()))
    return frame[col].fillna(frame[key].map(mode))

# ---- Engine.Type
df["Engine.Type"] = df["Engine.Type"].replace({"Geared Turbofan": "Turbo Fan"})
df["Engine.Type"] = df["Engine.Type"].where(~df["Engine.Type"].isin(["Unknown", "UNK", "NONE", "LR", "Hybrid Rocket"]))
df["Engine.Type"] = fill_with_group_mode(df, "Engine.Type").fillna("Unknown")

# ---- Number.of.Engines
df["Number.of.Engines"] = df["Number.of.Engines"].where(df["Number.of.Engines"] > 0)   # 0 -> NaN
df["Number.of.Engines"] = fill_with_group_mode(df, "Number.of.Engines")

# ---- Weather.Condition
df["Weather.Condition"] = df["Weather.Condition"].str.upper().where(lambda s: s.isin(["VMC", "IMC"])).fillna("Unknown")

# ---- Purpose.of.flight
purpose_map = {"Air Race show": "Air Race/show",
               "Public Aircraft - Federal": "Public Aircraft",
               "Public Aircraft - State": "Public Aircraft",
               "Public Aircraft - Local": "Public Aircraft",
               "Executive/corporate": "Business"}
df["Purpose.of.flight"] = df["Purpose.of.flight"].replace(purpose_map).fillna("Unknown")
rare = df["Purpose.of.flight"].value_counts()
rare = rare[rare < 50].index
df["Purpose.of.flight"] = df["Purpose.of.flight"].where(~df["Purpose.of.flight"].isin(rare), "Other")

# ---- Broad.phase.of.flight
df["Broad.phase.of.flight"] = df["Broad.phase.of.flight"].fillna("Unknown")

for c in other_cols:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")

--- Engine.Type ---
Engine.Type
Reciprocating    51990
Turbo Prop        2641
Turbo Fan         2234
Turbo Jet          337
Unknown            253
Turbo Shaft         13 

--- Weather.Condition ---
Weather.Condition
VMC        50025
IMC         4496
Unknown     2947 

--- Number.of.Engines ---
Number.of.Engines
1.0    48035
2.0     8728
3.0      299
4.0      208
NaN      198 

--- Purpose.of.flight ---
Purpose.of.flight
Personal              33389
Instructional          7907
Unknown                6894
Aerial Application     3154
Business               3004
Positioning             938
Other Work Use          502
Ferry                   488
Aerial Observation      370
Public Aircraft         342
Skydiving               172
Flight Test             121
Banner Tow               97
Other                    90 

--- Broad.phase.of.flight ---
Broad.phase.of.flight
Unknown        17128
Landing        11149
Takeoff         8258
Cruise          6747
Maneuvering     4269
Approach        4236
Taxi

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [23]:
df = df.drop(columns=["_key"])          # temporary helper column

non_null = df.notna().sum().sort_values()
print("Non-null counts (lowest first):")
print(non_null.head(12).to_string())

Non-null counts (lowest first):
Schedule                7163
Air.carrier            10445
FAR.Description        18222
Longitude              22307
Latitude               22312
Airport.Code           34905
Airport.Name           36473
Publication.Date       47860
Report.Status          53889
Aircraft.Damage        55933
Destroyed              55933
Registration.Number    56881


The instructions say to keep columns with **more than 20,000 non-null values**. Columns below that are mostly empty (`FAR.Description`, `Schedule`, `Air.carrier`) and are dropped. (`Latitude`/`Longitude` just clear the threshold, so they are kept by this rule; they are stored in mixed formats and are not needed for this analysis.)

In [24]:
MIN_NON_NULL = 20_000
dropped = non_null[non_null <= MIN_NON_NULL].index.tolist()
df = df.dropna(axis=1, thresh=MIN_NON_NULL + 1)

print("Dropped columns:", dropped)
print(f"\nFinal shape: {df.shape}")
print("Remaining NaNs:")
df.isna().sum()[lambda s: s > 0]

Dropped columns: ['Schedule', 'Air.carrier', 'FAR.Description']

Final shape: (57468, 33)
Remaining NaNs:


Location                  25
Country                   82
Latitude               35156
Longitude              35161
Airport.Code           22563
Airport.Name           20995
Aircraft.Damage         1535
Registration.Number      587
Number.of.Engines        198
Report.Status           3579
Publication.Date        9608
Destroyed               1535
dtype: int64

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [25]:
df["Event.Year"] = df["Event.Date"].dt.year      # handy for later analysis
df.to_csv("data/AviationData_cleaned.csv", index=False)

# quick check: reload and confirm
check = pd.read_csv("data/AviationData_cleaned.csv", parse_dates=["Event.Date"])
print("Saved to data/AviationData_cleaned.csv ->", check.shape)
check.head()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14412\4190404090.py:5: DtypeWarning: Columns (0: Latitude, 1: Longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  check = pd.read_csv("data/AviationData_cleaned.csv", parse_dates=["Event.Date"])


Saved to data/AviationData_cleaned.csv -> (57468, 34)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.Damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers,Total.Fatal.Serious,Fatal.Serious.Fraction,Destroyed,Make.Model,Event.Year
0,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,Non-Fatal,Substantial,Airplane,N1296M,CESSNA,182P,No,1.0,Reciprocating,Personal,0.0,0.0,1.0,3.0,VMC,Approach,Probable Cause,NaN,4.0,0.0,0.0,0.0,CESSNA 182P,1983
1,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,Non-Fatal,Substantial,Airplane,N2639C,CESSNA,182RG,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,0.0,CESSNA 182RG,1983
2,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,Non-Fatal,Substantial,Airplane,N58664,CESSNA,182P,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1.0,0.0,0.0,0.0,CESSNA 182P,1983
3,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,Non-Fatal,Substantial,Airplane,N3084R,PIPER,PA-28R-200,No,1.0,Reciprocating,Personal,0.0,0.0,2.0,0.0,VMC,Approach,Probable Cause,NaN,2.0,0.0,0.0,0.0,PIPER PA-28R-200,1983
4,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,NaN,NaN,NaN,SPRINGBROOK,Non-Fatal,Substantial,Airplane,N89399,CESSNA,140,No,1.0,Reciprocating,Instructional,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,2.0,0.0,0.0,0.0,CESSNA 140,1983
